# 1. Định nghĩa param grid cho từng model

In [31]:
from sklearn.model_selection import ParameterGrid

import sys
from pathlib import Path

# Thêm thư mục gốc project (cha của notebooks/) vào sys.path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
from src.models import evaluate_val_decision_tree, evaluate_val_random_forest, evaluate_val_adaboost, evaluate_val_logistic_regression
from src.data.loader import read_csv
from src.config import DATA_RAW_DIR, DATA_PROCESSED_DIR, DATA_FILTERED_DIR

param_grids = {
    "decision_tree": {
        "max_depth": [3, 5, 7, 10, None],
        "min_samples_leaf": [1, 3, 5, 10],
        "min_samples_split": [2, 5, 10],
    },
    "random_forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [5, 8, 12, None],
        "min_samples_leaf": [1, 3, 5],
        "max_features": ["sqrt", "log2"],
    },
    "adaboost": {
        "n_estimators": [50, 100, 150],
        "learning_rate": [0.1, 0.5, 1.0],
        "base_max_depth": [1, 2, 3],
    },
    "logistic_regression": {
        "C": [0.01, 0.1, 1, 10, 100],
        "solver": ["saga"],  # 'saga' hỗ trợ cả L1, L2, ElasticNet và Multiclass rất tốt
        "l1_ratio": [0.2, 0.5, 0.7],  # Chỉ áp dụng khi penalty='elasticnet'
        "max_iter": [500, 1000, 1500, 2000]
    }
}

# 2. Loop qua từng bộ tham số, đánh giá trên val, lưu kết quả

## 2.1 Original data

In [32]:
X_train, y_train = read_csv(DATA_RAW_DIR/"train.csv")
X_val, y_val = read_csv(DATA_RAW_DIR/"val.csv")

<class 'pandas.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   gender                      700 non-null    int64  
 1   study_time_hours            700 non-null    float64
 2   attendance_percent          700 non-null    float64
 3   sleep_hours                 700 non-null    float64
 4   parental_education          700 non-null    int64  
 5   internet_access             700 non-null    int64  
 6   extracurricular_activities  700 non-null    int64  
 7   part_time_job               700 non-null    int64  
 8   previous_grade              700 non-null    float64
 9   final_grade                 700 non-null    int64  
dtypes: float64(4), int64(6)
memory usage: 54.8 KB
None
final_grade
3    247
4    199
2    183
1     62
0      9
Name: count, dtype: int64
Shape df:  (700, 10)
Shape X:  (700, 9)
Shape y:  (700,)
<class 'pandas.DataFrame'>

In [33]:
def tune_model(evaluate_fn, param_grid, X_train, y_train, X_val, y_val, model_name):
    """Chạy grid search thủ công trên tập val, tắt print từng lần train (isprint=False)
    để tránh spam log khi thử nhiều tổ hợp tham số. Trả về dataframe kết quả, sort theo accuracy giảm dần.
    """
    results = []
    for params in ParameterGrid(param_grid):
        model, acc = evaluate_fn(X_train, y_train, X_val, y_val, isprint=False, **params)
        results.append({**params, "accuracy": acc})

    results_df = pd.DataFrame(results).sort_values("accuracy", ascending=False).reset_index(drop=True)
    print(f"\nTop 5 tham số tốt nhất cho {model_name}:")
    print(results_df.head())
    return results_df

In [34]:
dt_results = tune_model(evaluate_val_decision_tree, param_grids["decision_tree"], X_train, y_train, X_val, y_val, "Decision Tree")
rf_results = tune_model(evaluate_val_random_forest, param_grids["random_forest"], X_train, y_train, X_val, y_val, "Random Forest")
ada_results = tune_model(evaluate_val_adaboost, param_grids["adaboost"], X_train, y_train, X_val, y_val, "AdaBoost")
log_results = tune_model(evaluate_val_logistic_regression,param_grids["logistic_regression"], X_train, y_train, X_val, y_val, "Logistic Regression")


Top 5 tham số tốt nhất cho Decision Tree:
   max_depth  min_samples_leaf  min_samples_split  accuracy
0        7.0                 1                  5      0.53
1        7.0                 1                  2      0.52
2        7.0                 3                  2      0.52
3        7.0                 3                  5      0.52
4        5.0                10                  5      0.49

Top 5 tham số tốt nhất cho Random Forest:
   max_depth max_features  min_samples_leaf  n_estimators  accuracy
0        5.0         sqrt                 1           100       0.5
1        5.0         sqrt                 1           300       0.5
2        5.0         sqrt                 3           300       0.5
3        5.0         sqrt                 3           200       0.5
4        5.0         log2                 3           200       0.5

Top 5 tham số tốt nhất cho AdaBoost:
   base_max_depth  learning_rate  n_estimators  accuracy
0               2            0.5           150     

## 2.2 FE data

In [35]:
X_train_fe, y_train_fe = read_csv(DATA_PROCESSED_DIR/"train.csv")
X_val_fe, y_val_fe = read_csv(DATA_PROCESSED_DIR/"val.csv")

<class 'pandas.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   gender                       700 non-null    int64  
 1   study_time_hours             700 non-null    float64
 2   attendance_percent           700 non-null    float64
 3   sleep_hours                  700 non-null    float64
 4   parental_education           700 non-null    int64  
 5   internet_access              700 non-null    int64  
 6   extracurricular_activities   700 non-null    int64  
 7   part_time_job                700 non-null    int64  
 8   previous_grade               700 non-null    float64
 9   final_grade                  700 non-null    int64  
 10  study_efficiency             700 non-null    float64
 11  study_time_x_previous_grade  700 non-null    float64
 12  attendance_x_sleep           700 non-null    float64
 13  sleep_hours_group            70

In [36]:
dt_results = tune_model(evaluate_val_decision_tree, param_grids["decision_tree"], X_train_fe, y_train_fe, X_val_fe, y_val_fe, "Decision Tree")
rf_results = tune_model(evaluate_val_random_forest, param_grids["random_forest"], X_train_fe, y_train_fe, X_val_fe, y_val_fe, "Random Forest")
ada_results = tune_model(evaluate_val_adaboost, param_grids["adaboost"], X_train_fe, y_train_fe, X_val_fe, y_val_fe, "AdaBoost")
log_results = tune_model(evaluate_val_logistic_regression,param_grids["logistic_regression"], X_train_fe, y_train_fe, X_val_fe, y_val_fe, "Logistic Regression")


Top 5 tham số tốt nhất cho Decision Tree:
   max_depth  min_samples_leaf  min_samples_split  accuracy
0        7.0                 3                  2      0.49
1       10.0                 3                  5      0.49
2       10.0                 3                  2      0.49
3        7.0                 3                  5      0.49
4        NaN                 1                  5      0.48

Top 5 tham số tốt nhất cho Random Forest:
   max_depth max_features  min_samples_leaf  n_estimators  accuracy
0        8.0         log2                 1           100      0.57
1        8.0         sqrt                 1           100      0.57
2        8.0         sqrt                 5           200      0.56
3        8.0         sqrt                 5           100      0.56
4        8.0         log2                 5           200      0.56

Top 5 tham số tốt nhất cho AdaBoost:
   base_max_depth  learning_rate  n_estimators  accuracy
0               3            1.0           150     

d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: Con


Top 5 tham số tốt nhất cho Logistic Regression:
     C  l1_ratio  max_iter solver  accuracy
0  0.1       0.5       500   saga      0.66
1  0.1       0.5      1000   saga      0.66
2  0.1       0.5      1500   saga      0.66
3  0.1       0.5      2000   saga      0.66
4  0.1       0.7       500   saga      0.64


d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


## 2.3 Filtered data

In [37]:
X_train_filt, y_train_filt = read_csv(DATA_PROCESSED_DIR/"train.csv")
X_val_filt, y_val_filt = read_csv(DATA_PROCESSED_DIR/"val.csv")

<class 'pandas.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   gender                       700 non-null    int64  
 1   study_time_hours             700 non-null    float64
 2   attendance_percent           700 non-null    float64
 3   sleep_hours                  700 non-null    float64
 4   parental_education           700 non-null    int64  
 5   internet_access              700 non-null    int64  
 6   extracurricular_activities   700 non-null    int64  
 7   part_time_job                700 non-null    int64  
 8   previous_grade               700 non-null    float64
 9   final_grade                  700 non-null    int64  
 10  study_efficiency             700 non-null    float64
 11  study_time_x_previous_grade  700 non-null    float64
 12  attendance_x_sleep           700 non-null    float64
 13  sleep_hours_group            70

In [38]:
dt_filt_results = tune_model(evaluate_val_decision_tree, param_grids["decision_tree"], X_train_filt, y_train_filt, X_val_filt, y_val_filt, "Decision Tree")
rf_filt_results = tune_model(evaluate_val_random_forest, param_grids["random_forest"], X_train_filt, y_train_filt, X_val_filt, y_val_filt, "Random Forest")
ada_filt_results = tune_model(evaluate_val_adaboost, param_grids["adaboost"], X_train_filt, y_train_filt, X_val_filt, y_val_filt, "AdaBoost")
log_filt_results = tune_model(evaluate_val_logistic_regression,param_grids["logistic_regression"], X_train_filt, y_train_filt, X_val_filt, y_val_filt, "Logistic Regression")


Top 5 tham số tốt nhất cho Decision Tree:
   max_depth  min_samples_leaf  min_samples_split  accuracy
0        7.0                 3                  2      0.49
1       10.0                 3                  5      0.49
2       10.0                 3                  2      0.49
3        7.0                 3                  5      0.49
4        NaN                 1                  5      0.48

Top 5 tham số tốt nhất cho Random Forest:
   max_depth max_features  min_samples_leaf  n_estimators  accuracy
0        8.0         log2                 1           100      0.57
1        8.0         sqrt                 1           100      0.57
2        8.0         sqrt                 5           200      0.56
3        8.0         sqrt                 5           100      0.56
4        8.0         log2                 5           200      0.56

Top 5 tham số tốt nhất cho AdaBoost:
   base_max_depth  learning_rate  n_estimators  accuracy
0               3            1.0           150     

d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: Con


Top 5 tham số tốt nhất cho Logistic Regression:
     C  l1_ratio  max_iter solver  accuracy
0  0.1       0.5       500   saga      0.66
1  0.1       0.5      1000   saga      0.66
2  0.1       0.5      1500   saga      0.66
3  0.1       0.5      2000   saga      0.66
4  0.1       0.7       500   saga      0.64


d:\HCMUT\AIO\Project\data_env\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
